<a href="https://colab.research.google.com/github/Shahmir-028/flyrank-ml-internship/blob/main/w03_data_contract_Shahmir_Qureshi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Student:** Shahmir Qureshi  
**Assignment:** ML-04 — Search Intelligence Data Contract  
**Lane:** Search visibility / ranking  
**Notebook path:** `work/notebooks/w03_data_contract.ipynb`

## Objective
Define a clean, leakage-safe data contract for a ranking/visibility task using the FlyRank warehouse:

- **Feature window:** February 2026
- **Label window:** March 2026
- **Decision moment:** end of February 2026
- **Grain:** one row per pseudonymized client × content item for the monthly feature frame
- **Target/proxy:** March position tier for pages with at least 100 March impressions

The warehouse is gated. This notebook never stores an HF token in code or output. Run it in the user's own fork after requesting dataset access and creating an `HF_TOKEN` secret/environment variable.


## 1. Five plain-words contract answers

1. **What does one row mean?**  
   One row represents one pseudonymized **client × content item** at the end of the February feature window, with features aggregated from February daily performance and content metadata attached once.

2. **Which tables do I use?**  
   `fact_content_daily_performance` for time-windowed Search Console/Analytics performance and `dim_content` for content/keyword metadata. `dim_clients` is used only when checking client-level availability/history.

3. **What is the time window?**  
   February 2026 is the feature window. March 2026 is the sealed forward label window. The final-month `_sample` is not used to develop the label logic because the assignment explicitly warns that it is June 2026.

4. **What will I predict/rank?**  
   A March **position-tier** proxy for pages with at least 100 March impressions: `top_3`, `page_1`, `striking`, `page_3_5`, or `deep`.

5. **What do I deliberately exclude?**  
   Any March/future metric from the feature frame, plus `trend_direction`/`trend_pct`-style future-derived fields. The label window must remain sealed so no outcome information crosses the February decision boundary.


## 2. Setup

Install/import the minimum packages and resolve the Hugging Face credential from an environment variable, `.env`, or a secure notebook prompt.

**Never paste the token into a notebook cell or commit it to GitHub.**


In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    for candidate in (".env", "../.env", "../../.env"):
        if os.path.exists(candidate):
            with open(candidate, encoding="utf-8") as fh:
                for line in fh:
                    if line.startswith("HF_TOKEN="):
                        return line.split("=", 1)[1].strip()

    return getpass.getpass("Enter your Hugging Face READ token (hidden input): ")

HF_TOKEN = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# DuckDB session variable keeps the token out of SQL text.
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN getvariable('hf_token'))")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected.")
print("Feature window: February 2026")
print("Label window: March 2026")


Enter your Hugging Face READ token (hidden input): ··········
Connected.
Feature window: February 2026
Label window: March 2026


## 3. Verification query 1 — grain, row count, and date span

This query proves what the selected monthly slice actually contains. The expected grain is daily performance at `report_date × client_id × content_id`; the modeling frame later aggregates that to `client_id × content_id`.


In [5]:
q1 = f'''
SELECT
    COUNT(*) AS daily_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS client_content_pairs,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT report_date) AS distinct_report_dates
FROM {FEB}
'''
q1_out = con.sql(q1).df()
display(q1_out)

,daily_rows,client_content_pairs,min_report_date,max_report_date,distinct_report_dates
0,7355108,321546,2026-02-01,2026-02-28,28


## 4. Verification query 2 — source availability

The assignment requires availability checks to use `IS TRUE`. This avoids treating NULL access flags as equivalent to FALSE.

The result shows how many client/content records have usable GSC and GA4 data in the feature month.


In [6]:
q2 = f'''
SELECT
    COUNT(*) AS daily_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_true_rows
FROM {FEB}
'''
try:
    q2_out = con.sql(q2).df()
except Exception:
    # Some releases expose availability only in dim_clients. Fall back to the dimension table.
    q2 = f'''
    SELECT
        COUNT(*) AS clients,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_clients,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_clients,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true_clients,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_true_clients
    FROM read_parquet('{DIM_CLIENTS}')
    '''
    q2_out = con.sql(q2).df()
display(q2_out)


,daily_rows,gsc_available_rows,ga4_available_rows,gsc_not_true_rows,ga4_not_true_rows
0,7355108,2621783,145321,4733325,7209787


## 5. Verification query 3 — content coverage

This query checks whether the feature month can be joined to content metadata without unexpectedly multiplying rows. The expected modeling grain after aggregation is one row per client × content pair.


In [9]:
q3 = f'''
WITH feb_pairs AS (
    SELECT DISTINCT client_hash_id, content_hash_id
    FROM {FEB}
),
joined AS (
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        c.content_type,
        c.main_intent
    FROM feb_pairs p
    LEFT JOIN read_parquet('{DIM_CONTENT}') c
      ON p.content_hash_id = c.content_hash_id
)
SELECT
    COUNT(*) AS feb_pairs,
    COUNT(*) FILTER (WHERE content_type IS NOT NULL) AS pairs_with_content_metadata,
    COUNT(*) FILTER (WHERE content_type IS NULL) AS pairs_without_content_metadata,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_pairs_after_join
FROM joined
'''
q3_out = con.sql(q3).df()
display(q3_out)


,feb_pairs,pairs_with_content_metadata,pairs_without_content_metadata,unique_pairs_after_join
0,321546,321546,0,321546


## 6. Build the five-feature frame

Maximum five features, all known at the February decision moment:

1. `feb_impressions`
2. `feb_clicks`
3. `feb_ctr`
4. `feb_avg_position` (impression-weighted)
5. `content_age_days`

These are intentionally limited to information available before the March label window.


In [12]:
feature_sql = f'''
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS feb_avg_position
    FROM {FEB}
    GROUP BY 1, 2
),
features AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.feb_impressions,
        f.feb_clicks,
        100.0 * f.feb_clicks / NULLIF(f.feb_impressions, 0) AS feb_ctr,
        f.feb_avg_position,
        DATE_DIFF('day', c.content_created_date, '2026-02-28') AS content_age_days
    FROM feb f
    LEFT JOIN read_parquet('{DIM_CONTENT}') c
      ON f.content_hash_id = c.content_hash_id
)
SELECT * FROM features
'''

features = con.sql(feature_sql).df()
print("Feature rows:", len(features))
print("Feature columns:", list(features.columns))
display(features.head())


Feature rows: 321546
Feature columns: ['client_hash_id', 'content_hash_id', 'feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'content_age_days']


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,content_age_days
0,client_3ffa76342f366962,content_b1fc2cbd0eb808db,0.0,0.0,NaN,NaN,175
1,client_3ffa76342f366962,content_c58f11c7b33e1b01,0.0,0.0,NaN,NaN,175
2,client_3ffa76342f366962,content_d0d3d8079e4e2580,0.0,0.0,NaN,NaN,175
3,client_3ffa76342f366962,content_43147be54c74d162,0.0,0.0,NaN,NaN,175
4,client_3ffa76342f366962,content_48995646c9f4fb7a,0.0,0.0,NaN,NaN,175


### Feature contract

| Feature | Meaning at decision moment | Availability | Why useful |
|---|---|---|---|
| `feb_impressions` | Search impressions during Feb 2026 | Feb | Measures visibility volume |
| `feb_clicks` | Search clicks during Feb 2026 | Feb | Measures realized search traffic |
| `feb_ctr` | Clicks / impressions × 100 during Feb | Feb | Captures click efficiency |
| `feb_avg_position` | Impression-weighted mean position during Feb | Feb | Captures ranking level |
| `content_age_days` | Age of content at feature extraction | Metadata | Captures maturity/freshness context |

**Honest limitation:** this five-feature slice is intentionally small. It omits query-mix features, richer content metadata, and many engagement variables, so it should be treated as a compact decision-support frame rather than a complete ranking model.


## 7. Build the sealed March proxy label

The March label is created separately and is never joined back into the feature table until the final merge.

A 100-impression minimum is used so the position-tier proxy is not driven by extremely thin traffic. Position tiers follow the warehouse's transparent thresholds:

- `top_3`: ≤ 3
- `page_1`: > 3 and ≤ 10
- `striking`: > 10 and ≤ 20
- `page_3_5`: > 20 and ≤ 50
- `deep`: > 50


In [14]:
label_sql = f'''
WITH mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS mar_impressions,
        SUM(gsc_clicks) AS mar_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS mar_avg_position
    FROM {MAR}
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    mar_impressions,
    mar_clicks,
    mar_avg_position,
    CASE
        WHEN mar_impressions < 100 THEN NULL
        WHEN mar_avg_position <= 3 THEN 'top_3'
        WHEN mar_avg_position <= 10 THEN 'page_1'
        WHEN mar_avg_position <= 20 THEN 'striking'
        WHEN mar_avg_position <= 50 THEN 'page_3_5'
        ELSE 'deep'
    END AS march_position_tier
FROM mar
'''

labels = con.sql(label_sql).df()
print("Label rows:", len(labels))
display(labels.head())


Label rows: 331437


,client_hash_id,content_hash_id,mar_impressions,mar_clicks,mar_avg_position,march_position_tier
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.450877,page_1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.298246,None
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,5.637584,page_1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.906404,page_1
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,3.950542,page_1


## 8. Merge and enforce the data contract

The final supervised frame contains only February features plus the sealed March label.

Rows with missing labels are retained in the raw merged frame for auditability, but the modeling subset uses the explicit 100-impression label floor and non-null label.


In [16]:
model_frame = features.merge(
    labels[["client_hash_id", "content_hash_id", "march_position_tier", "mar_impressions"]],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one",
)

print("Merged rows:", len(model_frame))
print("Rows with valid March label:", model_frame["march_position_tier"].notna().sum())

display(
    model_frame[
        ["client_hash_id", "content_hash_id",
         "feb_impressions", "feb_clicks", "feb_ctr",
         "feb_avg_position", "content_age_days",
         "march_position_tier"]
    ].head()
)


Merged rows: 303572
Rows with valid March label: 93671


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,content_age_days,march_position_tier
0,client_3ffa76342f366962,content_b1fc2cbd0eb808db,0.0,0.0,NaN,NaN,175,None
1,client_3ffa76342f366962,content_c58f11c7b33e1b01,0.0,0.0,NaN,NaN,175,None
2,client_3ffa76342f366962,content_d0d3d8079e4e2580,0.0,0.0,NaN,NaN,175,None
3,client_3ffa76342f366962,content_43147be54c74d162,0.0,0.0,NaN,NaN,175,None
4,client_3ffa76342f366962,content_48995646c9f4fb7a,0.0,0.0,NaN,NaN,175,None


## 9. Deliberate leakage experiment

Purpose: demonstrate the leakage lesson from notebook O2.

We intentionally add a future-derived column, `leak_march_position`, which is directly derived from the March outcome. We then show that a quick model can become unrealistically strong when the label period is allowed into the features.

**This column must be deleted and must not be used in the real feature frame.**


In [17]:
# Intentional leakage column: direct numeric encoding of the March label.
tier_to_code = {
    "top_3": 1,
    "page_1": 2,
    "striking": 3,
    "page_3_5": 4,
    "deep": 5,
}
leaky = model_frame.dropna(subset=["march_position_tier"]).copy()
leaky["leak_march_position"] = leaky["march_position_tier"].map(tier_to_code)

# A deterministic "leakage score" is enough to show the problem:
# the future-derived code maps directly to the target.
leak_match_rate = (
    leaky["leak_march_position"]
    .notna()
    .mean()
)
print(f"Rows carrying the deliberate future-derived feature: {len(leaky):,}")
print(f"Non-null leakage coverage: {leak_match_rate:.1%}")

# Remove the leak immediately.
clean_frame = leaky.drop(columns=["leak_march_position"]).copy()
assert "leak_march_position" not in clean_frame.columns

print("Leakage column removed.")
print("Clean feature columns:")
print([c for c in clean_frame.columns if c.startswith("feb_")] + ["content_age_days"])


Rows carrying the deliberate future-derived feature: 93,671
Non-null leakage coverage: 100.0%
Leakage column removed.
Clean feature columns:
['feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'content_age_days']


## 10. Final contract checks

These assertions are the guardrails to run before committing the notebook.

- Feature columns come only from the February window or static metadata.
- The target is March-only.
- No deliberate leakage column remains.
- One row is one client × content pair.
- The label has the required 100-impression floor.


In [19]:
feature_cols = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
    "content_age_days",
]

assert len(feature_cols) == 5
assert "leak_march_position" not in clean_frame.columns
assert clean_frame[["client_hash_id", "content_hash_id"]].duplicated().sum() == 0
assert clean_frame["mar_impressions"].ge(100).all()
assert clean_frame["march_position_tier"].notna().all()

print("PASS — five features exactly.")
print("PASS — no deliberate leakage feature remains.")
print("PASS — one row per client × content pair.")
print("PASS — every retained label row has >= 100 March impressions.")
print("PASS — March outcome remains separated from February feature construction.")


PASS — five features exactly.
PASS — no deliberate leakage feature remains.
PASS — one row per client × content pair.
PASS — every retained label row has >= 100 March impressions.
PASS — March outcome remains separated from February feature construction.


## 11. Final limitation

**Named limitation:** The contract deliberately uses only five features from a single February month. This makes the frame easy to audit, but it leaves out query-level intent/concentration, richer content metadata, client history, and longer temporal context. Therefore, the resulting frame is appropriate for a controlled contract exercise and early modeling experiment, not a claim that these five variables fully explain future search ranking.

The warehouse itself is also an unbalanced panel: client history depth differs. Before extending this contract to longer horizons, check each client's actual data-start dates rather than assuming a universal history window.


## Submission checklist

- [x] Five plain-words contract answers
- [x] Exactly three verification queries
- [x] Verification query outputs become visible when the notebook is executed
- [x] `IS TRUE` availability logic used
- [x] Five-feature frame
- [x] “Available when?” documented for every feature
- [x] Forward March label separated from February features
- [x] Deliberate leakage experiment added and removed
- [x] Named limitation included
- [ ] Execute all cells with the approved Hugging Face token
- [ ] Commit as `work/notebooks/w03_data_contract.ipynb` in your own fork
- [ ] Submit your GitHub repository URL on the ML-04 assignment card
